# CUSUM Fixed h Grid Viewer

Explore fixed-threshold, mean/std CUSUM settings over `k` and `h`. The CUSUM accumulation itself is used as the persistence mechanism, so no additional consecutive-point rule is applied.


In [1]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown, clear_output

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXPERIMENTS_DIR = PROJECT_ROOT / "results" / "kelmarsh" / "experiments"
CUSUM_C = 1

def list_experiments():
    if not EXPERIMENTS_DIR.exists():
        return []
    return sorted(p.name for p in EXPERIMENTS_DIR.iterdir() if (p / "metadata.json").exists())

def format_number(value):
    return f"{float(value):g}".replace(".", "p")

def grid_label(k_values, h_values):
    ks = [float(v.strip()) for v in k_values.split(",") if v.strip()]
    hs = [float(v.strip()) for v in h_values.split(",") if v.strip()]
    return f"fixed_meanstd_no_consecutive_k{format_number(min(ks))}-{format_number(max(ks))}_h{format_number(min(hs))}-{format_number(max(hs))}"

def grid_path(run_id, k_values, h_values):
    return EXPERIMENTS_DIR / run_id / f"cusum_grid_{grid_label(k_values, h_values)}.csv"

def run_grid(run_id, k_values, h_values, force=False):
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "experiments" / "main_grid_cusum_fixed_meanstd.py"),
        "--run-id", run_id,
        "--k-values", k_values,
        "--h-values", h_values,
    ]
    if force:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("CUSUM grid run failed")

def run_selected_setting(run_id, k, h):
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "main_detect_cusum.py"),
        "--run-id", run_id,
        "--scale-type", "std",
        "--reference-value", str(k),
        "--decision-threshold", str(h),
    ]
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("CUSUM selected setting run failed")

def load_grid(run_id, k_values, h_values):
    path = grid_path(run_id, k_values, h_values)
    if not path.exists():
        raise FileNotFoundError(f"Grid summary not found: {path}")
    return pd.read_csv(path)

def plot_heatmap(ax, data, value_col, title, cmap, fmt=".1%"):
    pivot = data.pivot(index="h", columns="k", values=value_col).sort_index(ascending=False)
    image = ax.imshow(pivot.values, aspect="auto", cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel("k: reference value")
    ax.set_ylabel("h: decision threshold")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{v:g}" for v in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{v:g}" for v in pivot.index])
    for row_i, h in enumerate(pivot.index):
        for col_i, k in enumerate(pivot.columns):
            value = pivot.loc[h, k]
            label = format(value, fmt) if pd.notna(value) else ""
            ax.text(col_i, row_i, label, ha="center", va="center", fontsize=8, color="black")
    plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

def add_f1_columns(data):
    data = data.copy()
    if "recall" not in data.columns:
        data["recall"] = data["detection_rate"]
    if "precision" not in data.columns:
        denominator = data["true_alarm_episodes"] + data["operational_false_alarm_episodes"]
        data["precision"] = (data["true_alarm_episodes"] / denominator).fillna(0)
    if "f1_score" not in data.columns:
        denominator = data["precision"] + data["recall"]
        data["f1_score"] = (2 * data["precision"] * data["recall"] / denominator).fillna(0)
    return data

def show_grid(run_id, k_values, h_values):
    data = add_f1_columns(load_grid(run_id, k_values, h_values))
    display(Markdown("### Best Settings by F1 Score"))
    display(data.sort_values(["f1_score", "operational_false_alarm_rate"], ascending=[False, True])[[
        "k", "h", "precision", "recall", "f1_score", "detected_events", "total_events",
        "operational_false_alarm_rate", "total_alarm_episodes",
        "true_alarm_episodes", "operational_false_alarm_episodes",
    ]].head(15).style.format({
        "precision": "{:.2%}",
        "recall": "{:.2%}",
        "f1_score": "{:.2%}",
        "operational_false_alarm_rate": "{:.2%}",
    }))
    fig, axes = plt.subplots(1, 3, figsize=(19, 5), constrained_layout=True)
    plot_heatmap(axes[0], data, "f1_score", "F1 score", "YlGnBu")
    plot_heatmap(axes[1], data, "detection_rate", "Recall: event detection rate", "Greens")
    plot_heatmap(axes[2], data, "operational_false_alarm_rate", "Operational FA rate", "Reds")
    plt.show()


In [2]:
runs = list_experiments()
if not runs:
    raise FileNotFoundError(f"No experiment metadata found in {EXPERIMENTS_DIR}")

if HAS_WIDGETS:
    run_dropdown = widgets.Dropdown(options=runs, value=runs[-1], description="Run")
    k_text = widgets.Text(value="0,1,2,3,4,5,6,7,8,9,10", description="k values", layout=widgets.Layout(width="520px"))
    h_text = widgets.Text(value="5,10,15,20,25", description="h values", layout=widgets.Layout(width="360px"))
    force_checkbox = widgets.Checkbox(value=False, description="force recompute")
    run_grid_button = widgets.Button(description="Run/refresh grid", button_style="primary")
    load_grid_button = widgets.Button(description="Load heatmaps")
    selected_k = widgets.FloatText(value=2.0, description="selected k")
    selected_h = widgets.FloatText(value=10.0, description="selected h")
    run_one_button = widgets.Button(description="Run selected setting", button_style="warning")
    output = widgets.Output()

    def on_run_grid(_):
        with output:
            clear_output(wait=True)
            run_grid(run_dropdown.value, k_text.value, h_text.value, force=force_checkbox.value)
            show_grid(run_dropdown.value, k_text.value, h_text.value)

    def on_load_grid(_):
        with output:
            clear_output(wait=True)
            show_grid(run_dropdown.value, k_text.value, h_text.value)

    def on_run_one(_):
        with output:
            clear_output(wait=True)
            run_selected_setting(run_dropdown.value, selected_k.value, selected_h.value)

    run_grid_button.on_click(on_run_grid)
    load_grid_button.on_click(on_load_grid)
    run_one_button.on_click(on_run_one)
    display(widgets.VBox([
        widgets.HBox([run_dropdown]),
        widgets.HBox([k_text, h_text]),
        widgets.HBox([run_grid_button, load_grid_button, force_checkbox]),
        widgets.HBox([selected_k, selected_h, run_one_button]),
        output,
    ]))
else:
    print("ipywidgets is not available. Use run_grid(...) and show_grid(...) manually.")
